In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input, Normalization
import matplotlib.pyplot as plt

from adaptive_latents import datasets
import numpy as np
from adaptive_latents.utils import resample_matched_timeseries


In [ ]:
d = datasets.Odoherty21Dataset(bin_width=0.03)  # (timesteps x neurons)

In [ ]:
from adaptive_latents import StreamingKalmanFilter
from adaptive_latents.sim_stim import make_sr
from itertools import cycle

kwargs = dict(
        autoreg=StreamingKalmanFilter,
        isi_generator=cycle([1]),
        stim_rate=None,
        exit_time=np.inf,
        decay_rate=.8,
        prosvd_k=10,
        max_l0_norm=30,
        attempt_correction=True,
        heed_stimuli=True,
        stim_time_delay=0,
        regressor_stim_delay=0,
        optimization_method='jaxopt',
        u_to_s_model_type='identity',
        design_type=None,
        true_S='identity',
        stim_timing_method='random',
        n_identity_prior=10,
        stim_direction_type='first',
        initial_nostim_period=5,
        stim_reg_maxlen=500,
        smoothing_tau=None,
        centerer_init_size=0,
        last_dim_red='prosvd',
        show_tqdm=True,
)

_, _, log_nostim = make_sr(
        d.neural_data,
        np.random.default_rng(0),
        stim_magnitude=0,
        **kwargs
)

_, _, log_stim = make_sr(
        d.neural_data,
        np.random.default_rng(0),
        stim_magnitude=100,
        **kwargs
)



preturbed_neural_data = log_nostim['high_d_with_stim']
unpreturbed_neural_data = log_stim['high_d_with_stim']

beh = resample_matched_timeseries(d.behavioral_data, d.behavioral_data.t, unpreturbed_neural_data.t)  # (timesteps x 2)

normalized_beh = (beh - np.nanmean(beh, axis=0)) / np.nanstd(beh, axis=0)

In [ ]:
n = Normalization()
n.adapt(beh[None,:,:])

model = Sequential(
    [
        Input(shape=unpreturbed_neural_data.shape),
        LSTM(200, return_sequences=True),
        Dense(2),
    ]
)

model.compile(optimizer='adam', loss='mse')
model.fit(unpreturbed_neural_data[None, :,:], normalized_beh[None,:,:], epochs=300, verbose=True)

In [ ]:
%matplotlib inline

pred = model.predict(unpreturbed_neural_data[None,:,:]).squeeze()
plt.scatter(beh.flatten(), pred.flatten(), s=5, c=np.arange(pred.size), cmap='plasma')


In [ ]:
unpreturbed_pred = model.predict(unpreturbed_neural_data[None,:,:]).squeeze()*beh.std(axis=0) + beh.mean(axis=0)
preturbed_pred = model.predict(preturbed_neural_data[None,:,:]).squeeze()*beh.std(axis=0) + beh.mean(axis=0)

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
# plt.plot(beh.t, beh[:,0])
plt.plot(beh.t, unpreturbed_pred[:,0])
plt.plot(beh.t, preturbed_pred[:,0])

plt.xlim([000, 56])

for t in log_stim['stim_intended_samples'].t:
    plt.axvline(t, color='red', alpha=0.3)


In [ ]:
from adaptive_latents import StimRegressor
sr_aware = StimRegressor(log_level=2)
sr_unaware = StimRegressor(log_level=2, heed_stimuli=False, attempt_correction=False)

stims = log_stim['stims']
stims.t -= 1e-10

sr_aware.offline_run_on([(preturbed_pred, 'X'), (stims, 'stim')], show_tqdm=True)
sr_unaware.offline_run_on([(preturbed_pred, 'X'), (stims, 'stim')], show_tqdm=True)

In [ ]:
fig, ax = plt.subplots()
from adaptive_latents import ArrayWithTime

e = ArrayWithTime.from_list(sr_unaware.log['pred_error'], squeeze_type='to_2d')
ax.plot(e.t,np.linalg.norm(e, axis=1))
print(np.nanmean(np.linalg.norm(e, axis=1)))

e = ArrayWithTime.from_list(sr_aware.log['pred_error'], squeeze_type='to_2d')
ax.plot(e.t,np.linalg.norm(e, axis=1))
print(np.nanmean(np.linalg.norm(e, axis=1)))

# sr.log.keys()